COMPARATIVE EVALUATION OF SCALABLE MULTI-CLASS CLASSIFICATION MODELS FOR NYPD ARREST SEVERITY PREDICTION USING DISTRIBUTED PYSPARK AND TABLEAU ANALYTICS

**1) Data Acquisition & Environment Setup**

**Cell 1: Spark Installation**

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-8/openjdk-8-jre-headless_8u472-ga-1%7e22.04_amd64.deb  404  Not Found [IP: 91.189.92.23 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-8/openjdk-8-jdk-headless_8u472-ga-1%7e22.04_amd64.deb  404  Not Found [IP: 91.189.92.23 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?


In [2]:
!java -version

openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


**Cell 2: Pip Instll Pyspark**

In [4]:
!pip install pyspark==3.5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 19.3 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=d5e766ce2a3641b1d1ffe34936e06f844e522b3fb0f2ba919635389ff63b065f
  Stored in directory: /root/.cache/pip/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

In [5]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/usr/local/lib/python3.10/dist-packages/pyspark"

**CELL 3** Start SparkSession (Big Data tuning)

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYC_Cleaning")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.network.timeout", "600s")
    .config("spark.executor.heartbeatInterval", "60s")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.1


**CELL 4**  Upload again (confirm correct filename)


In [ ]:
from google.colab import files
uploaded = files.upload()
print(uploaded.keys())

Saving NYPD_Arrests_Data__Historic_.csv to NYPD_Arrests_Data__Historic_.csv
dict_keys(['NYPD_Arrests_Data__Historic_.csv'])


**CELL 5**  Confirm file exists in /content



In [ ]:
!ls -lh /content | sed -n '1,200p'

total 1.2G
-rw-r--r-- 1 root root 1.2G Feb  9 18:28 NYPD_Arrests_Data__Historic_.csv
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data


**CELL 6**  **Robust file path detection**



In [ ]:
import os, glob

# Most likely exact path
CSV_PATH = "/content/NYPD_Arrests_Data_Historic_.csv"

# If it uploaded with a slightly different name, auto-pick a matching file
if not os.path.exists(CSV_PATH):
    matches = glob.glob("/content/*NYPD*Arrests*Historic*.csv")
    print("Matches found:", matches)
    if matches:
        CSV_PATH = matches[0]

print("Using CSV_PATH:", CSV_PATH)
print("Exists?", os.path.exists(CSV_PATH))

Matches found: ['/content/NYPD_Arrests_Data__Historic_.csv']
Using CSV_PATH: /content/NYPD_Arrests_Data__Historic_.csv
Exists? True


**CELL 7**  Install PySpark again (safe repeat)



In [ ]:
!pip -q install pyspark==3.5.1

**CELL 8** Start SparkSession again (stable runtime)



In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYPD_Arrests_Cleaning")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.network.timeout", "600s")
    .config("spark.executor.heartbeatInterval", "60s")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.1


**2) Data Ingestion + Initial Profiling**

**CELL 9**  Read CSV into Spark DataFrame


In [ ]:
from pyspark.sql import functions as F

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", "\"")
    .csv(CSV_PATH)
)

print("Rows:", df_raw.count())
print("Cols:", len(df_raw.columns))
df_raw.printSchema()
df_raw.show(5, truncate=False)

Rows: 5986025
Cols: 19
root
 |-- ARREST_KEY: integer (nullable = true)
 |-- ARREST_DATE: string (nullable = true)
 |-- PD_CD: integer (nullable = true)
 |-- PD_DESC: string (nullable = true)
 |-- KY_CD: integer (nullable = true)
 |-- OFNS_DESC: string (nullable = true)
 |-- LAW_CODE: string (nullable = true)
 |-- LAW_CAT_CD: string (nullable = true)
 |-- ARREST_BORO: string (nullable = true)
 |-- ARREST_PRECINCT: integer (nullable = true)
 |-- JURISDICTION_CODE: integer (nullable = true)
 |-- AGE_GROUP: string (nullable = true)
 |-- PERP_SEX: string (nullable = true)
 |-- PERP_RACE: string (nullable = true)
 |-- X_COORD_CD: double (nullable = true)
 |-- Y_COORD_CD: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Lon_Lat: string (nullable = true)

+----------+-----------+-----+-----------------+-----+--------------+----------+----------+-----------+---------------+-----------------+---------+--------+--------------+---------

**CELL 10**  Standardize column names



In [ ]:
import re

def clean_colname(c: str) -> str:
    c = c.strip()
    c = re.sub(r"\s+", "_", c)
    c = re.sub(r"[^0-9a-zA-Z_]", "", c)
    return c

df = df_raw.toDF(*[clean_colname(c) for c in df_raw.columns])
print(df.columns)

['ARREST_KEY', 'ARREST_DATE', 'PD_CD', 'PD_DESC', 'KY_CD', 'OFNS_DESC', 'LAW_CODE', 'LAW_CAT_CD', 'ARREST_BORO', 'ARREST_PRECINCT', 'JURISDICTION_CODE', 'AGE_GROUP', 'PERP_SEX', 'PERP_RACE', 'X_COORD_CD', 'Y_COORD_CD', 'Latitude', 'Longitude', 'Lon_Lat']


**CELL 11**  Convert empty strings to NULL + quick null snapshot



In [ ]:
for c in df.columns:
    df = df.withColumn(
        c,
        F.when(F.trim(F.col(c).cast("string")) == "", F.lit(None)).otherwise(F.col(c))
    )

# quick null count snapshot
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show(truncate=False)

+----------+-----------+-----+-------+-----+---------+--------+----------+-----------+---------------+-----------------+---------+--------+---------+----------+----------+--------+---------+-------+
|ARREST_KEY|ARREST_DATE|PD_CD|PD_DESC|KY_CD|OFNS_DESC|LAW_CODE|LAW_CAT_CD|ARREST_BORO|ARREST_PRECINCT|JURISDICTION_CODE|AGE_GROUP|PERP_SEX|PERP_RACE|X_COORD_CD|Y_COORD_CD|Latitude|Longitude|Lon_Lat|
+----------+-----------+-----+-------+-----+---------+--------+----------+-----------+---------------+-----------------+---------+--------+---------+----------+----------+--------+---------+-------+
|0         |0          |884  |9169   |9788 |9169     |196     |24990     |8          |0              |10               |17       |0       |0        |1         |1         |5       |5        |5      |
+----------+-----------+-----+-------+-----+---------+--------+----------+-----------+---------------+-----------------+---------+--------+---------+----------+----------+--------+---------+-------+



**CELL 12** Parse date + enforce numeric types



In [ ]:
# Date parsing (NYPD arrest data usually supports to_date directly)
if "ARREST_DATE" in df.columns:
    df = df.withColumn("ARREST_DATE", F.to_date(F.col("ARREST_DATE")))

# Cast numeric fields safely
for c in ["PD_CD","KY_CD","ARREST_PRECINCT","JURISDICTION_CODE"]:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("int"))

for c in ["X_COORD_CD","Y_COORD_CD","Latitude","Longitude"]:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("double"))

df.printSchema()

root
 |-- ARREST_KEY: integer (nullable = true)
 |-- ARREST_DATE: date (nullable = true)
 |-- PD_CD: integer (nullable = true)
 |-- PD_DESC: string (nullable = true)
 |-- KY_CD: integer (nullable = true)
 |-- OFNS_DESC: string (nullable = true)
 |-- LAW_CODE: string (nullable = true)
 |-- LAW_CAT_CD: string (nullable = true)
 |-- ARREST_BORO: string (nullable = true)
 |-- ARREST_PRECINCT: integer (nullable = true)
 |-- JURISDICTION_CODE: integer (nullable = true)
 |-- AGE_GROUP: string (nullable = true)
 |-- PERP_SEX: string (nullable = true)
 |-- PERP_RACE: string (nullable = true)
 |-- X_COORD_CD: double (nullable = true)
 |-- Y_COORD_CD: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Lon_Lat: string (nullable = true)



**CELL 13** Remove duplicates


In [ ]:
before = df.count()

if "ARREST_KEY" in df.columns:
    df = df.dropDuplicates(["ARREST_KEY"])
else:
    df = df.dropDuplicates()

after = df.count()
print("Before:", before, "| After:", after, "| Removed:", before - after)

Before: 5986025 | After: 5986025 | Removed: 0


**3) Feature Engineering for a Clear Target**

**CELL 14** Create the target label (SEVERITY) from LAW_CAT_CD



In [ ]:
# LAW_CAT_CD is typically: F (Felony), M (Misdemeanor), V (Violation)
if "LAW_CAT_CD" in df.columns:
    df = df.withColumn(
        "SEVERITY",
        F.when(F.upper(F.col("LAW_CAT_CD")).isin("F", "FELONY"), "FELONY")
         .when(F.upper(F.col("LAW_CAT_CD")).isin("M", "MISDEMEANOR"), "MISDEMEANOR")
         .when(F.upper(F.col("LAW_CAT_CD")).isin("V", "VIOLATION"), "VIOLATION")
         .otherwise("OTHER")
    )

df.select([c for c in ["LAW_CAT_CD","SEVERITY"] if c in df.columns]).show(20, truncate=False)

+----------+-----------+
|LAW_CAT_CD|SEVERITY   |
+----------+-----------+
|F         |FELONY     |
|M         |MISDEMEANOR|
|I         |OTHER      |
|F         |FELONY     |
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|F         |FELONY     |
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|F         |FELONY     |
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|F         |FELONY     |
|F         |FELONY     |
|F         |FELONY     |
+----------+-----------+
only showing top 20 rows



**CELL 15** Coordinate sanity filtering



In [ ]:
if "Latitude" in df.columns and "Longitude" in df.columns:
    df = df.filter(
        (F.col("Latitude").isNull() | ((F.col("Latitude") >= -90) & (F.col("Latitude") <= 90))) &
        (F.col("Longitude").isNull() | ((F.col("Longitude") >= -180) & (F.col("Longitude") <= 180)))
    )

print("Rows after coord filter:", df.count())

Rows after coord filter: 5986025


**4) Tableau-Ready Clean Export**

**CELL 16** Write cleaned dataset to a single CSV folder



In [ ]:
OUT_DIR = "/content/NYPD_Tableau_Cleaned"
(
    df.coalesce(1)
      .write
      .mode("overwrite")
      .option("header", "true")
      .csv(OUT_DIR)
)

!ls -lh {OUT_DIR}

total 1.3G
-rw-r--r-- 1 root root 1.3G Feb  9 19:11 part-00000-3268e452-c19a-4ce9-b2b0-ab93e1129c3e-c000.csv
-rw-r--r-- 1 root root    0 Feb  9 19:11 _SUCCESS


**CELL 17** Copy Spark part file into a single named CSV



In [ ]:
import glob, shutil

part_files = glob.glob(f"{OUT_DIR}/part-*.csv")
print("Part files:", part_files)

FINAL_CSV = "/content/NYPD_Tableau_Cleaned.csv"
shutil.copy(part_files[0], FINAL_CSV)

!ls -lh /content | grep NYPD_Tableau_Cleaned.csv

Part files: ['/content/NYPD_Tableau_Cleaned/part-00000-3268e452-c19a-4ce9-b2b0-ab93e1129c3e-c000.csv']
-rw-r--r-- 1 root root 1.3G Feb  9 19:12 NYPD_Tableau_Cleaned.csv


**CELL 18** Download cleaned CSV



In [ ]:
from google.colab import files
files.download(FINAL_CSV)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**5) ML Dataset Preparation**

In [ ]:
from pyspark.sql import functions as F

# I keep only rows where my target (SEVERITY) exists
df_ml = df.filter(F.col("SEVERITY").isNotNull())

# I standardize text columns I will use (safe cleanup)
for c in ["ARREST_BORO", "AGE_GROUP", "PERP_SEX", "PERP_RACE", "OFNS_DESC"]:
    if c in df_ml.columns:
        df_ml = df_ml.withColumn(c, F.upper(F.trim(F.col(c))))

# I also make sure my target is clean
df_ml = df_ml.withColumn("SEVERITY", F.upper(F.trim(F.col("SEVERITY"))))

print("Rows for ML:", df_ml.count())
df_ml.groupBy("SEVERITY").count().orderBy(F.desc("count")).show(truncate=False)

Rows for ML: 5986025
+-----------+-------+
|SEVERITY   |count  |
+-----------+-------+
|MISDEMEANOR|3866398|
|FELONY     |1767834|
|VIOLATION  |297792 |
|OTHER      |54001  |
+-----------+-------+



**CELL 20** Select feature columns (categorical + numeric)



In [ ]:
# I define my feature columns (categorical + numeric) based on the dataset columns that exist
candidate_categoricals = ["ARREST_BORO", "AGE_GROUP", "PERP_SEX", "PERP_RACE", "OFNS_DESC"]
candidate_numerics     = ["PD_CD", "KY_CD", "ARREST_PRECINCT", "JURISDICTION_CODE",
                          "X_COORD_CD", "Y_COORD_CD", "Latitude", "Longitude"]

cat_cols = [c for c in candidate_categoricals if c in df_ml.columns]
num_cols = [c for c in candidate_numerics if c in df_ml.columns]

print("Categorical features:", cat_cols)
print("Numeric features:", num_cols)

# I keep only the needed columns
keep_cols = ["SEVERITY"] + cat_cols + num_cols
df_ml = df_ml.select(*keep_cols)

df_ml.show(5, truncate=False)

Categorical features: ['ARREST_BORO', 'AGE_GROUP', 'PERP_SEX', 'PERP_RACE', 'OFNS_DESC']
Numeric features: ['PD_CD', 'KY_CD', 'ARREST_PRECINCT', 'JURISDICTION_CODE', 'X_COORD_CD', 'Y_COORD_CD', 'Latitude', 'Longitude']
+-----------+-----------+---------+--------+--------------+-----------------+-----+-----+---------------+-----------------+----------+----------+------------------+------------------+
|SEVERITY   |ARREST_BORO|AGE_GROUP|PERP_SEX|PERP_RACE     |OFNS_DESC        |PD_CD|KY_CD|ARREST_PRECINCT|JURISDICTION_CODE|X_COORD_CD|Y_COORD_CD|Latitude          |Longitude         |
+-----------+-----------+---------+--------+--------------+-----------------+-----+-----+---------------+-----------------+----------+----------+------------------+------------------+
|MISDEMEANOR|B          |18-24    |M       |BLACK         |DANGEROUS DRUGS  |567  |235  |44             |0                |1006490.0 |244533.0  |40.837842121000044|-73.91962775199994|
|MISDEMEANOR|B          |18-24    |M       |W

**CELL 21** Train/Test split + caching (big data best practice)



In [ ]:
from pyspark.storagelevel import StorageLevel

# I split my data into train and test for fair evaluation
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

# I cache for speed (big data best practice)
train_df = train_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df  = test_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Train:", train_df.count(), "Test:", test_df.count())

Train: 4788218 Test: 1197807


**6) Feature Pipeline (Encoding + Scaling)**

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

LABEL_COL = "SEVERITY"

# I create my label indexer (multiclass)
label_indexer = StringIndexer(
    inputCol=LABEL_COL,
    outputCol="label",
    handleInvalid="keep"
)

# I index + one-hot encode categoricals
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in cat_cols],
    outputCols=[f"{c}_ohe" for c in cat_cols],
    handleInvalid="keep"
)

# I assemble numeric + encoded features into one vector
assembler_inputs = [f"{c}_ohe" for c in cat_cols] + num_cols

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features_raw",
    handleInvalid="keep"
)

# I scale features (helps LR/SVM/MLP)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False
)

# I fit the shared pipeline ONCE on train (fairness + speed)
feat_pipe = Pipeline(stages=[label_indexer] + indexers + [encoder, assembler, scaler]).fit(train_df)

train_ready = feat_pipe.transform(train_df).select("label", "features")
test_ready  = feat_pipe.transform(test_df).select("label", "features")

print("Prepared train rows:", train_ready.count(), "Prepared test rows:", test_ready.count())
train_ready.show(3, truncate=False)

Prepared train rows: 4788218 Prepared test rows: 1197807
+-----+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|features                                                                                                                                                                                                                       |
+-----+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1.0  |(197,[2,7,86,89,98,189,190,191,192,193,194,195,196],[2.3805691760696672,2.3120345345313744,2.670022823005776,2.0008815451476267,2.54245191864113,NaN,NaN,1.2202481144957833,NaN,49.792267197794324,1.6041012475572365,NaN,NaN])|
|1.0  |(197,[2,

**7) Model Training Setup (4 models required)**

**CELL 23** Define evaluators + define 4 models (LR, DT, SVM, Deep Learning)



In [ ]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, LinearSVC, OneVsRest, MultilayerPerceptronClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# I define my evaluators
eval_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
eval_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
eval_wp  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
eval_wr  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

# I count number of classes for MLP
num_classes = int(train_ready.select("label").distinct().count())
print("Number of classes:", num_classes)

models = {}

# 1) Logistic Regression
models["Logistic Regression"] = LogisticRegression(
    featuresCol="features", labelCol="label",
    maxIter=50, regParam=0.0, elasticNetParam=0.0
)

# 2) Decision Tree
models["Decision Tree"] = DecisionTreeClassifier(
    featuresCol="features", labelCol="label",
    maxDepth=10, minInstancesPerNode=50
)

# 3) SVM (multiclass using OneVsRest + LinearSVC)
svm_binary = LinearSVC(featuresCol="features", labelCol="label", maxIter=50, regParam=0.1)
models["SVM (OneVsRest LinearSVC)"] = OneVsRest(classifier=svm_binary, labelCol="label", featuresCol="features")

# 4) Deep Learning (Multilayer Perceptron)
# I set a simple architecture: input -> hidden -> hidden -> output
input_size = train_ready.select("features").first()["features"].size
layers = [input_size, 64, 32, num_classes]
models["Deep Learning (MLP)"] = MultilayerPerceptronClassifier(
    featuresCol="features", labelCol="label",
    layers=layers, maxIter=50, seed=42, blockSize=256
)

print("MLP layers:", layers)

Number of classes: 4
MLP layers: [197, 64, 32, 4]


**8) Train All 4 Models + Evaluate + Comparison Table**

**CELL 24** Train and evaluate all models + build comparison table


In [ ]:
from pyspark.sql import Row

results = []

for name, model in models.items():
    print("\n==============================")
    print("Training:", name)
    print("==============================")

    fitted = model.fit(train_ready)
    preds  = fitted.transform(test_ready).cache()

    acc = float(eval_acc.evaluate(preds))
    f1  = float(eval_f1.evaluate(preds))
    wp  = float(eval_wp.evaluate(preds))
    wr  = float(eval_wr.evaluate(preds))

    print(f"{name} -> Accuracy: {acc:.4f} | F1: {f1:.4f} | W-Precision: {wp:.4f} | W-Recall: {wr:.4f}")

    results.append(Row(Model=name, Accuracy=acc, F1=f1, WeightedPrecision=wp, WeightedRecall=wr))

    preds.unpersist()

results_df = spark.createDataFrame(results).orderBy(F.desc("F1"))
results_df.show(truncate=False)


Training: Logistic Regression


Py4JJavaError: An error occurred while calling o1224.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 123.0 failed 1 times, most recent failure: Lost task 0.0 in stage 123.0 (TID 2853) (6e5f65dbc397 executor driver): java.lang.RuntimeException: Vector values MUST NOT be NaN or Infinity, but got (197,[2,7,86,89,98,189,190,191,192,193,194,195,196],[2.3805691760696672,2.3120345345313744,2.670022823005776,2.0008815451476267,2.54245191864113,NaN,NaN,1.2202481144957833,NaN,49.792267197794324,1.6041012475572365,NaN,NaN])
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.project_doConsume_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartit

**9) Numeric Data Quality Check + Fix (important for stability)**

**CELL 25** Check NaN/Infinity in numeric columns


In [ ]:
from pyspark.sql import functions as F

# I quickly check NaN / Infinity in numeric columns
checks = []
for c in num_cols:
    checks.append(
        F.sum(F.when(F.isnan(F.col(c)) | F.col(c).isin(float("inf"), float("-inf")), 1).otherwise(0)).alias(c)
    )

nan_inf_counts = df_ml.select(checks)
nan_inf_counts.show(truncate=False)

+-----+-----+---------------+-----------------+----------+----------+--------+---------+
|PD_CD|KY_CD|ARREST_PRECINCT|JURISDICTION_CODE|X_COORD_CD|Y_COORD_CD|Latitude|Longitude|
+-----+-----+---------------+-----------------+----------+----------+--------+---------+
|0    |0    |0              |0                |0         |0         |0       |0        |
+-----+-----+---------------+-----------------+----------+----------+--------+---------+



**CELL 26** Replace NaN/Inf and fill missing numeric values with medians



In [ ]:
from pyspark.sql import functions as F

df_clean = df_ml

# 1) I replace NaN/Infinity in numeric columns with NULL
for c in num_cols:
    df_clean = df_clean.withColumn(
        c,
        F.when(F.col(c).isin(float("inf"), float("-inf")), None)
         .when(F.isnan(F.col(c)), None)
         .otherwise(F.col(c))
    )

# 2) I fill NULL numeric values with the median (robust) or 0 if you prefer
# Median in Spark: approxQuantile
fill_map = {}
for c in num_cols:
    med = df_clean.approxQuantile(c, [0.5], 0.01)[0]  # 1% relative error
    if med is None:
        med = 0.0
    fill_map[c] = float(med)

df_clean = df_clean.fillna(fill_map)

print("Numeric fill values (medians):")
for k,v in fill_map.items():
    print(k, "->", v)

Numeric fill values (medians):
PD_CD -> 503.0
KY_CD -> 341.0
ARREST_PRECINCT -> 60.0
JURISDICTION_CODE -> 0.0
X_COORD_CD -> 1004950.0
Y_COORD_CD -> 208800.0
Latitude -> 40.74088057900008
Longitude -> -73.92539159499995


**10) Rebuild Train/Test + Feature Pipeline After Cleaning**

**CELL 27** Re-split cleaned data and cache



In [ ]:
from pyspark.storagelevel import StorageLevel

train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)

train_df = train_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df  = test_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Train:", train_df.count(), "Test:", test_df.count())

Train: 4788218 Test: 1197807


**CELL 28** Refit feature pipeline after numeric cleaning



In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

LABEL_COL = "SEVERITY"

label_indexer = StringIndexer(inputCol=LABEL_COL, outputCol="label", handleInvalid="keep")

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in cat_cols],
    outputCols=[f"{c}_ohe" for c in cat_cols],
    handleInvalid="keep"
)

assembler_inputs = [f"{c}_ohe" for c in cat_cols] + num_cols

assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features_raw", handleInvalid="keep")

scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)

feat_pipe = Pipeline(stages=[label_indexer] + indexers + [encoder, assembler, scaler]).fit(train_df)

train_ready = feat_pipe.transform(train_df).select("label", "features")
test_ready  = feat_pipe.transform(test_df).select("label", "features")

print("Prepared train rows:", train_ready.count(), "Prepared test rows:", test_ready.count())

Prepared train rows: 4788218 Prepared test rows: 1197807


**CELL 29** Verify no NaN/Inf inside the feature vector


In [ ]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

# I verify that my feature vectors contain no NaN/Inf
arr_df = train_ready.select(vector_to_array("features").alias("fa"))

bad = arr_df.select(
    F.sum(
        F.expr("aggregate(fa, 0, (acc, x) -> acc + IF(isnan(x) OR x = double('inf') OR x = -double('inf'), 1, 0))")
    ).alias("bad_values")
).collect()[0]["bad_values"]

print("Bad values in train features:", bad)

Bad values in train features: 0


**11) Final Training Run + Final Comparison Table**

**CELL 30 R**etrain 4 models and produce final metrics table


In [ ]:
from pyspark.sql import Row
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, LinearSVC, OneVsRest, MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import functions as F

# Evaluators
eval_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
eval_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
eval_wp  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
eval_wr  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

num_classes = int(train_ready.select("label").distinct().count())
input_size = train_ready.select("features").first()["features"].size
layers = [input_size, 64, 32, num_classes]
print("Classes:", num_classes, " | MLP layers:", layers)

models = {}
models["Logistic Regression"] = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
models["Decision Tree"] = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=10, minInstancesPerNode=50)

svm_binary = LinearSVC(featuresCol="features", labelCol="label", maxIter=50, regParam=0.1)
models["SVM (OneVsRest LinearSVC)"] = OneVsRest(classifier=svm_binary, labelCol="label", featuresCol="features")

models["Deep Learning (MLP)"] = MultilayerPerceptronClassifier(
    featuresCol="features", labelCol="label",
    layers=layers, maxIter=50, seed=42, blockSize=256
)

results = []

for name, model in models.items():
    print("\n==============================")
    print("Training:", name)
    print("==============================")

    fitted = model.fit(train_ready)
    preds  = fitted.transform(test_ready).cache()

    acc = float(eval_acc.evaluate(preds))
    f1  = float(eval_f1.evaluate(preds))
    wp  = float(eval_wp.evaluate(preds))
    wr  = float(eval_wr.evaluate(preds))

    print(f"{name} -> Accuracy: {acc:.4f} | F1: {f1:.4f} | W-Precision: {wp:.4f} | W-Recall: {wr:.4f}")

    results.append(Row(Model=name, Accuracy=acc, F1=f1, WeightedPrecision=wp, WeightedRecall=wr))

    preds.unpersist()

results_df = spark.createDataFrame(results).orderBy(F.desc("F1"))
results_df.show(truncate=False)

Classes: 4  | MLP layers: [197, 64, 32, 4]

Training: Logistic Regression
Logistic Regression -> Accuracy: 0.9929 | F1: 0.9926 | W-Precision: 0.9930 | W-Recall: 0.9929

Training: Decision Tree
Decision Tree -> Accuracy: 0.9932 | F1: 0.9930 | W-Precision: 0.9931 | W-Recall: 0.9932

Training: SVM (OneVsRest LinearSVC)
SVM (OneVsRest LinearSVC) -> Accuracy: 0.9167 | F1: 0.9131 | W-Precision: 0.9247 | W-Recall: 0.9167

Training: Deep Learning (MLP)
Deep Learning (MLP) -> Accuracy: 0.6463 | F1: 0.5074 | W-Precision: 0.4177 | W-Recall: 0.6463
+-------------------------+------------------+------------------+------------------+------------------+
|Model                    |Accuracy          |F1                |WeightedPrecision |WeightedRecall    |
+-------------------------+------------------+------------------+------------------+------------------+
|Decision Tree            |0.9931683484901992|0.9929643067073213|0.9930656850715233|0.9931683484901992|
|Logistic Regression      |0.992942101690

**12) Confusion Matrix for Best Model**

**CELL 31** Identify best model + confusion matrix counts



In [ ]:
best_model_name = results_df.first()["Model"]
print("Best model by F1:", best_model_name)

best_fitted = models[best_model_name].fit(train_ready)

best_preds = (
    best_fitted.transform(test_ready)
              .select("label", "prediction")
              .cache()
)

# Confusion matrix counts: label vs prediction
confusion_df = (
    best_preds.groupBy("label", "prediction")
              .count()
              .orderBy("label", "prediction")
)

confusion_df.show(200, truncate=False)

best_preds.unpersist()

Best model by F1: Decision Tree
+-----+----------+------+
|label|prediction|count |
+-----+----------+------+
|0.0  |0.0       |773312|
|0.0  |1.0       |606   |
|0.0  |3.0       |196   |
|1.0  |0.0       |3720  |
|1.0  |1.0       |349634|
|1.0  |3.0       |9     |
|2.0  |0.0       |187   |
|2.0  |2.0       |58919 |
|2.0  |3.0       |331   |
|3.0  |0.0       |2840  |
|3.0  |2.0       |294   |
|3.0  |3.0       |7759  |
+-----+----------+------+



DataFrame[label: double, prediction: double]

**13) Export Results for Report / Tableau**

**CELL 32** Export model comparison table + confusion matrix table to CSV



In [ ]:
import glob, shutil, os

# --- 1) Export model comparison results ---
OUT_RESULTS_DIR = "/content/model_comparison_results"

(
    results_df.coalesce(1)
              .write.mode("overwrite")
              .option("header", "true")
              .csv(OUT_RESULTS_DIR)
)

part_results = glob.glob(f"{OUT_RESULTS_DIR}/part-*.csv")[0]
FINAL_RESULTS_CSV = "/content/model_comparison_results.csv"
shutil.copy(part_results, FINAL_RESULTS_CSV)

print("Saved model comparison CSV:", FINAL_RESULTS_CSV)


# --- 2) Export confusion matrix
OUT_CONF_DIR = "/content/confusion_matrix_counts"

try:
    (
        confusion_df.coalesce(1)
                    .write.mode("overwrite")
                    .option("header", "true")
                    .csv(OUT_CONF_DIR)
    )

    part_conf = glob.glob(f"{OUT_CONF_DIR}/part-*.csv")[0]
    FINAL_CONF_CSV = "/content/confusion_matrix_counts.csv"
    shutil.copy(part_conf, FINAL_CONF_CSV)

    print("Saved confusion matrix CSV:", FINAL_CONF_CSV)

except NameError:
    print("confusion_df not found. Please run CELL 7 first if you want the confusion matrix export.")


# --- 3) Show files in /content so I can download them ---
!ls -lh /content | sed -n '1,200p'

Saved model comparison CSV: /content/model_comparison_results.csv
Saved confusion matrix CSV: /content/confusion_matrix_counts.csv
total 2.5G
drwxr-xr-x 2 root root 4.0K Feb 10 01:53 confusion_matrix_counts
-rw-r--r-- 1 root root  176 Feb 10 01:53 confusion_matrix_counts.csv
drwxr-xr-x 2 root root 4.0K Feb 10 01:53 model_comparison_results
-rw-r--r-- 1 root root  430 Feb 10 01:53 model_comparison_results.csv
-rw-r--r-- 1 root root 1.2G Feb  9 18:28 NYPD_Arrests_Data__Historic_.csv
drwxr-xr-x 2 root root 4.0K Feb  9 19:11 NYPD_Tableau_Cleaned
-rw-r--r-- 1 root root 1.3G Feb  9 19:12 NYPD_Tableau_Cleaned.csv
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data


**CELL 33** Download model comparison CSV


In [ ]:
from google.colab import files
files.download("/content/model_comparison_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Create a Direct Colab Model Comparism CSV Table**

In [ ]:
import pandas as pd

data = {
    "Model": [
        "Decision Tree",
        "Logistic Regression",
        "SVM (OneVsRest LinearSVC)",
        "Deep Learning (MLP)"
    ],
    "Accuracy": [
        0.9931683484901992,
        0.99294210169084,
        0.9167152972056433,
        0.6462752346580042
    ],
    "F1": [
        0.9929643067073213,
        0.9925716602114856,
        0.9130844253402577,
        0.5074148109477526
    ],
    "WeightedPrecision": [
        0.9930656850715233,
        0.992965313352576,
        0.9246960137674678,
        0.4176725671786969
    ],
    "WeightedRecall": [
        0.9931683484901992,
        0.99294210169084,
        0.9167152972056434,
        0.6462752346580042
    ]
}

df = pd.DataFrame(data)
df.to_csv("model_results.csv", index=False)

print("CSV file created successfully!")

CSV file created successfully!


In [ ]:
from google.colab import files
files.download("model_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pyspark import StorageLevel

train_df, test_df = df_feat.randomSplit([0.8, 0.2], seed=42)
train_df = train_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df  = test_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Train:", train_df.count(), "Test:", test_df.count())

Train: 4789063 Test: 1196962


In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=50)

lr_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.0, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_grid,
    evaluator=evaluator_f1,
    numFolds=3,
    parallelism=2,
    seed=42
)

lr_cv_model = lr_cv.fit(train_df)
best_lr = lr_cv_model.bestModel

print("LR best regParam:", best_lr.getRegParam())
print("LR best elasticNetParam:", best_lr.getElasticNetParam())
print("LR best CV F1:", float(max(lr_cv_model.avgMetrics)))

lr_test_preds = best_lr.transform(test_df)
print("LR test F1:", float(evaluator_f1.evaluate(lr_test_preds)))
print("LR test ACC:", float(evaluator_acc.evaluate(lr_test_preds)))

LR best regParam: 0.0
LR best elasticNetParam: 0.0
LR best CV F1: 0.9936149031284978
LR test F1: 0.9937391523874562
LR test ACC: 0.9939371508869955
